# レッスン 11 - Model Context Protocol (MCP)

The **Model Context Protocol (MCP)** は、エージェントが実行時にツール、リソース、データソースを動的に発見して利用できるようにするオープン標準です。エージェントにツールをハードコーディングする代わりに、MCPはエージェントが必要に応じて機能を公開する外部サーバーに接続できるようにします。

このレッスンで学ぶこと:
- MCPが何であるか、そしてエージェントシステムにとってなぜ重要か
- MCPのクライアント・サーバーアーキテクチャがどのように機能するか
- MCPスタイルのツール探索を利用するエージェントの構築方法


## セットアップ

**前提条件:**
- デプロイ済みのモデルを含む Azure AI Foundry プロジェクト
- 認証のために `az login` を実行する


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Model Context Protocol（MCP）とは何ですか？

MCPは、AIエージェントが外部のツールやデータソースを発見し、相互作用するための標準的な方法を定義します:

- **MCP Server**: 標準プロトコルを通じてツール、リソース、プロンプトを公開する
- **MCP Client**: サーバーに接続し、利用可能な機能を検出するエージェントランタイム
- **Dynamic Discovery**: エージェントはハードコードされたツールを必要とせず — 実行時に利用可能なものを発見する

これは、エージェントのコードを変更せずに新しい機能を追加できる拡張可能なエージェントシステムを構築する際に強力です。


## MCPの仕組み

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. エージェント (MCPクライアント) がMCPサーバーに接続する
2. サーバーは利用可能なツールとそのスキーマの一覧で応答する
3. エージェントは推論中に検出された任意のツールを呼び出すことができる
4. 結果は同じプロトコルを通じて戻される


## MCPツールの検出をシミュレート

実際の MCP サーバーは稼働中のサーバープロセスを必要とするため、ここでは MCP に接続された宿泊サービスが提供するものを模擬する `@tool` 関数を使ってこのパターンを示します。

本番環境では、これらのツールはローカルに定義されるのではなく、MCP サーバーから動的に検出されます。


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCPスタイルのツールを用いたエージェントの構築


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## 本番環境での MCP

本番環境では、MCP は強力なパターンを可能にします:

- **動的なツール検出**: エージェントは MCP サーバーに接続し、実行時にツールを検出します
- **疎結合アーキテクチャ**: ツール提供者はエージェントとは独立して更新できます
- **組織間共有**: チームは MCP サーバー経由で機能を公開でき、どのエージェントでも利用できます
- **Microsoft Agent Framework のサポート**: MAF は `mcp` 統合を通じて組み込みの MCP クライアントサポートを提供します

MAF で実際の MCP サーバーを使用するには、`hosted_mcp_tool()` または MCP クライアント統合経由で接続します。

**詳しくはこちら:**
- [MCP仕様](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework の MCP サポート](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Summary

このレッスンでは、以下を学びました:
- **MCP** はエージェントとツール提供者間の動的ツール検出のためのオープン標準です
- **クライアント-サーバーアーキテクチャ** により、エージェントは実行時に機能を検出できます
- MCP は **拡張性があり疎結合なエージェントシステム** を可能にし、そこではコード変更なしでツールを追加できます
- Microsoft Agent Framework は本番利用のための **組み込みのMCPサポート** を提供します


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
免責事項：
この文書は AI 翻訳サービス「Co-op Translator」（https://github.com/Azure/co-op-translator）を用いて翻訳されました。正確性には努めていますが、自動翻訳には誤りや不正確な箇所が含まれる可能性がありますのでご注意ください。原文（元の言語による文書）を正式な情報源としてご確認ください。重要な情報については専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈の相違についても、当方は責任を負いません。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
